# Module 3 Homework — Stock Markets Analytics Zoomcamp 2026

Lesson reference: [`03-modeling/[2026]_Module_03_Colab_Time_Series_Modeling.ipynb`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/03-modeling/%5B2026%5D_Module_03_Colab_Time_Series_Modeling.ipynb)

Homework spec: [`cohorts/2026/homework3.md`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework3.md)
Submission form: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw03

## Data prep (shared)

Reproduces the lesson's `new_df`: data from 2000-01-01, `ln_volume`, dummies for `Month`, `Weekday`, `Ticker`, `ticker_type` **plus the new `month_wom`**, the 70/15/15 temporal split, and the hand rules `pred0`–`pred4` (Q2's two new rules included). Same cleaning as the lesson for the trees (±inf → NaN → 0).

In [1]:
"""
Shared data prep for Module 3: reproduces the lesson notebook's new_df
(dummies, temporal split, hand-rule predictions) with the homework extensions:
the month_wom dummies (Q1) and hand rules pred3/pred4 (Q2).

The course parquet is cached to data/ after the first download.
"""
from pathlib import Path

import gdown
import numpy as np
import pandas as pd

FILE_ID = "1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5"
DATA = Path("data") / "stocks_df_combined.parquet.brotli"

TARGET = "is_positive_growth_30d_future"

OHLCV = ["Open", "High", "Low", "Close", "Adj Close_x", "Volume"]
CATEGORICAL = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]
CUSTOM_NUMERICAL = ["SMA10", "SMA20", "growing_moving_average", "high_minus_low_relative", "volatility", "ln_volume"]
TECHNICAL_INDICATORS = [
    "adx", "adxr", "apo", "aroon_1", "aroon_2", "aroonosc",
    "bop", "cci", "cmo", "dx", "macd", "macdsignal", "macdhist", "macd_ext",
    "macdsignal_ext", "macdhist_ext", "macd_fix", "macdsignal_fix",
    "macdhist_fix", "mfi", "minus_di", "mom", "plus_di", "dm", "ppo",
    "roc", "rocp", "rocr", "rocr100", "rsi", "slowk", "slowd", "fastk",
    "fastd", "fastk_rsi", "fastd_rsi", "trix", "ultosc", "willr",
    "ad", "adosc", "obv", "atr", "natr", "ht_dcperiod", "ht_dcphase",
    "ht_phasor_inphase", "ht_phasor_quadrature", "ht_sine_sine", "ht_sine_leadsine",
    "ht_trendmod", "avgprice", "medprice", "typprice", "wclprice",
]
MACRO = ["gdppot_us_yoy", "gdppot_us_qoq", "cpi_core_yoy", "cpi_core_mom", "FEDFUNDS", "DGS1", "DGS5", "DGS10"]


def load_df_full():
    if not DATA.exists():
        DATA.parent.mkdir(exist_ok=True)
        gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", str(DATA), quiet=False)
    return pd.read_parquet(DATA)


def numerical_features(df):
    growth = [k for k in df.keys() if k.startswith("growth_") and "future" not in k]
    patterns = [k for k in df.keys() if "cdl" in k]
    return growth + TECHNICAL_INDICATORS + patterns + CUSTOM_NUMERICAL + MACRO


def temporal_split(df, train_prop=0.7, val_prop=0.15):
    min_date, max_date = df["Date"].min(), df["Date"].max()
    days = (max_date - min_date).days
    train_end = min_date + pd.Timedelta(days=days * train_prop)
    val_end = train_end + pd.Timedelta(days=days * val_prop)
    return np.where(df["Date"] <= train_end, "train", np.where(df["Date"] <= val_end, "validation", "test"))


def add_hand_rules(df):
    df["pred0_manual_cci"] = (df.cci > 200).astype(int)
    df["pred1_manual_prev_g1"] = (df.growth_30d > 1).astype(int)
    df["pred2_manual_prev_g1_and_snp"] = ((df.growth_30d > 1) & (df.growth_snp500_30d > 1)).astype(int)
    df["pred3_manual_dgs10_5"] = ((df.DGS10 <= 4.5) & (df.DGS5 <= 4)).astype(int)
    df["pred4_manual_dgs10_fedfunds"] = ((df.DGS10 > 4) & (df.FEDFUNDS <= 4.795)).astype(int)
    return df


def build_new_df():
    """Returns (new_df, NUMERICAL, DUMMIES) like the lesson notebook, plus month_wom dummies."""
    df_full = load_df_full()
    df_full["ln_volume"] = df_full.Volume.replace(0, np.nan).fillna(1e-9).apply(np.log)
    df = df_full[df_full.Date >= "2000-01-01"].copy()

    df["Weekday"] = df["Weekday"].astype(str)
    df["Month"] = df["Month"].dt.month.astype(str)
    df["month_wom"] = df["Date"].dt.month_name() + "_w" + ((df["Date"].dt.day - 1) // 7 + 1).astype(str)

    dummies = pd.get_dummies(df[CATEGORICAL], dtype="int32")
    new_df = pd.concat([df, dummies], axis=1)
    new_df["split"] = temporal_split(new_df)
    new_df = add_hand_rules(new_df.copy())
    return new_df, numerical_features(df_full), dummies.columns.to_list()


def clean_X(df, features):
    """Same cleaning as the lesson: +-inf -> NaN -> 0."""
    return df[features].replace([np.inf, -np.inf], np.nan).fillna(0)


def precision_by_prediction(new_df, predictions, split="test"):
    """Lesson-style precision: share of correct outcomes among rows where pred == 1."""
    sub = new_df[new_df.split == split]
    rows = []
    for p in predictions:
        positives = sub[sub[p] == 1]
        precision = (positives[TARGET] == 1).mean() if len(positives) else np.nan
        rows.append({"prediction": p, "positives": len(positives), "precision": precision})
    return pd.DataFrame(rows).set_index("prediction")

## Question 1: Dummies for Month and Week-of-Month

**Main task:** What is the absolute correlation value of the most correlated `<month>_w<week_of_month>` dummy with `is_positive_growth_30d_future`?

Week of month = `(day - 1) // 7 + 1`, combined with the month name (e.g. `October_w1`). Gives 115 dummies total, 60 of them `month_wom` — matching the spec.

In [2]:
"""
Question 1: Dummies for Month and Week-of-Month

What is the ABSOLUTE CORRELATION VALUE of the most correlated dummy variable
<month>_w<week_of_month> with the binary outcome is_positive_growth_30d_future?
"""
import pandas as pd



def main():
    new_df, numerical, dummies = build_new_df()
    wom_dummies = [d for d in dummies if d.startswith("month_wom_")]
    print(f"Dummies: {len(dummies)} total (spec ~115), {len(wom_dummies)} month_wom (spec ~60)")

    corr = new_df[numerical + dummies + [TARGET]].corr()[TARGET]
    corr_wom = pd.DataFrame(corr[wom_dummies])
    corr_wom["abs_corr"] = corr_wom[TARGET].abs()
    corr_wom = corr_wom.sort_values("abs_corr", ascending=False)
    print("\nTop 10 month_wom dummies by |corr|:")
    print(corr_wom.head(10))

    top = corr_wom.index[0]
    print(f"\nANSWER Q1: {top} -> |corr| = {corr_wom['abs_corr'].iloc[0]:.3f}")


if __name__ == "__main__":
    main()

Dummies: 115 total (spec ~115), 60 month_wom (spec ~60)



Top 10 month_wom dummies by |corr|:
                       is_positive_growth_30d_future  abs_corr
month_wom_October_w4                        0.024584  0.024584
month_wom_November_w3                       0.023251  0.023251
month_wom_February_w1                      -0.020861  0.020861
month_wom_January_w2                       -0.019846  0.019846
month_wom_January_w5                       -0.019792  0.019792
month_wom_November_w2                       0.019103  0.019103
month_wom_January_w3                       -0.018537  0.018537
month_wom_February_w2                      -0.018340  0.018340
month_wom_January_w4                       -0.018265  0.018265
month_wom_October_w3                        0.017856  0.017856

ANSWER Q1: month_wom_October_w4 -> |corr| = 0.025


## Question 2: New "Hand" Rules on Macro Variables

**Main task:** What is the precision score for the best of the new predictions (`pred3` or `pred4`)?

The spec says only one of the rules makes positive predictions on TEST — that's left over from the 2025 version. With the 2026 data (test = 2022-09-16 → 2026-09-18) both rules fire on ~48% of test rows, so the answer is the better of the two.

In [3]:
"""
Question 2: Define New "Hand" Rules on Macro and Technical Indicator Variables

What is the precision score for the best of the NEW predictions (pred3 or
pred4), rounded to 3 digits after the comma?
"""


def main():
    new_df, _, _ = build_new_df()
    predictions = [k for k in new_df.keys() if k.startswith("pred")]

    stats = precision_by_prediction(new_df, predictions)
    print("TEST set precision of hand rules:")
    print(stats)

    new = stats.loc[["pred3_manual_dgs10_5", "pred4_manual_dgs10_fedfunds"]].dropna()
    best = new["precision"].idxmax()
    print(f"\nANSWER Q2: {best} -> precision = {new.loc[best, 'precision']:.3f}")


if __name__ == "__main__":
    main()

TEST set precision of hand rules:
                              positives  precision
prediction                                        
pred0_manual_cci                    921   0.564604
pred1_manual_prev_g1              19158   0.578662
pred2_manual_prev_g1_and_snp      15244   0.572553
pred3_manual_dgs10_5              15910   0.587744
pred4_manual_dgs10_fedfunds       15921   0.502544

ANSWER Q2: pred3_manual_dgs10_5 -> precision = 0.588


## Question 3: Unique Correct Predictions from `pred5_clf_10`

**Main task:** How many TEST records have `pred5_clf_10` correct while all hand rules (`pred0`–`pred4`) are wrong?

Tree fit on TRAIN+VALIDATION with `random_state=42`, predicted on all rows. Uses the "advanced" row-wise function, so it generalises to any prediction column.

In [4]:
"""
Question 3: Unique Correct Predictions from a 10-Level Decision Tree Classifier

What is the total number of records in the TEST dataset where the new
prediction pred5_clf_10 is correct, while all 'hand' rule predictions
(pred0 to pred4) are incorrect?
"""
from sklearn.tree import DecisionTreeClassifier



def fit_and_predict_all(new_df, features, max_depth):
    """Fit on TRAIN+VALIDATION, predict on every row of new_df (same order)."""
    train = new_df.split.isin(["train", "validation"])
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf.fit(clean_X(new_df[train], features), new_df.loc[train, TARGET])
    return clf, clf.predict(clean_X(new_df, features))


def only_correct(row, pred, predictions):
    """True when `pred` is correct and every other prediction is wrong."""
    return row[pred] == row[TARGET] and all(row[p] != row[TARGET] for p in predictions if p != pred)


def main():
    new_df, numerical, dummies = build_new_df()
    hand_rules = [k for k in new_df.keys() if k.startswith("pred")]

    _, new_df["pred5_clf_10"] = fit_and_predict_all(new_df, numerical + dummies, max_depth=10)
    predictions = hand_rules + ["pred5_clf_10"]
    print("TEST set precision:")
    print(precision_by_prediction(new_df, predictions))

    test = new_df[new_df.split == "test"].copy()
    test["only_pred5_is_correct"] = test.apply(only_correct, axis=1, args=("pred5_clf_10", predictions)).astype(int)
    print(f"\nANSWER Q3: {test['only_pred5_is_correct'].sum()} TEST rows where only pred5_clf_10 is correct")


if __name__ == "__main__":
    main()

TEST set precision:
                              positives  precision
prediction                                        
pred0_manual_cci                    921   0.564604
pred1_manual_prev_g1              19158   0.578662
pred2_manual_prev_g1_and_snp      15244   0.572553
pred3_manual_dgs10_5              15910   0.587744
pred4_manual_dgs10_fedfunds       15921   0.502544
pred5_clf_10                       8108   0.597681



ANSWER Q3: 1219 TEST rows where only pred5_clf_10 is correct


## Question 4: Hyperparameter Tuning for a Decision Tree

**Main task:** What is the optimal tree depth (1 to 12)?

Each tree is fit on TRAIN+VALIDATION; precision is measured on TEST. Train+val precision keeps rising with depth while test precision peaks early — classic overfitting. Takes ~2 min.

In [5]:
"""
Question 4: Hyperparameter tuning for a Decision Tree

What is the optimal tree depth (from 1 to 12) for a DecisionTreeClassifier?

"""
import pandas as pd
from sklearn.metrics import precision_score
from sklearn.tree import export_text


DEPTHS = range(1, 13)


def main():
    new_df, numerical, dummies = build_new_df()
    features = numerical + dummies
    is_test = new_df.split == "test"

    rows = []
    for depth in DEPTHS:
        clf, y_pred = fit_and_predict_all(new_df, features, max_depth=depth)
        if depth <= 3:
            print(f"--- depth {depth} tree head ---")
            print(export_text(clf, feature_names=features, max_depth=3))
        rows.append({
            "max_depth": depth,
            "train_val_precision": precision_score(new_df.loc[~is_test, TARGET], y_pred[~is_test]),
            "test_precision": precision_score(new_df.loc[is_test, TARGET], y_pred[is_test], zero_division=0),
            "test_positives": int(y_pred[is_test].sum()),
        })
    scores = pd.DataFrame(rows).set_index("max_depth")
    print(scores)

    best_max_depth = scores["test_precision"].idxmax()
    _, new_df["pred5_clf_10"] = fit_and_predict_all(new_df, features, max_depth=10)
    _, new_df["pred6_clf_best"] = fit_and_predict_all(new_df, features, max_depth=best_max_depth)

    predictions = [k for k in new_df.keys() if k.startswith("pred")]
    print("\nTEST set precision, all predictions:")
    print(precision_by_prediction(new_df, predictions))

    print(f"\nANSWER Q4: best_max_depth = {best_max_depth} "
          f"(test precision = {scores.loc[best_max_depth, 'test_precision']:.4f})")


if __name__ == "__main__":
    main()

--- depth 1 tree head ---
|--- cpi_core_yoy <= 0.02
|   |--- class: 1
|--- cpi_core_yoy >  0.02
|   |--- class: 1



--- depth 2 tree head ---
|--- cpi_core_yoy <= 0.02
|   |--- DGS10 <= 1.75
|   |   |--- class: 1
|   |--- DGS10 >  1.75
|   |   |--- class: 1
|--- cpi_core_yoy >  0.02
|   |--- FEDFUNDS <= 4.96
|   |   |--- class: 0
|   |--- FEDFUNDS >  4.96
|   |   |--- class: 1



--- depth 3 tree head ---
|--- cpi_core_yoy <= 0.02
|   |--- DGS10 <= 1.75
|   |   |--- DGS1 <= 0.56
|   |   |   |--- class: 1
|   |   |--- DGS1 >  0.56
|   |   |   |--- class: 0
|   |--- DGS10 >  1.75
|   |   |--- growth_wti_oil_365d <= 1.50
|   |   |   |--- class: 1
|   |   |--- growth_wti_oil_365d >  1.50
|   |   |   |--- class: 1
|--- cpi_core_yoy >  0.02
|   |--- FEDFUNDS <= 4.96
|   |   |--- growth_btc_usd_365d <= 1.21
|   |   |   |--- class: 0
|   |   |--- growth_btc_usd_365d >  1.21
|   |   |   |--- class: 1
|   |--- FEDFUNDS >  4.96
|   |   |--- growth_wti_oil_365d <= 0.91
|   |   |   |--- class: 0
|   |   |--- growth_wti_oil_365d >  0.91
|   |   |   |--- class: 1



           train_val_precision  test_precision  test_positives
max_depth                                                     
1                     0.593358        0.570093           33163
2                     0.624742        0.622534           11760
3                     0.630933        0.571969           14263
4                     0.634836        0.629175            6677
5                     0.647208        0.599973            7487
6                     0.655503        0.606205            6704
7                     0.669198        0.621072            6682
8                     0.679946        0.617370           10420
9                     0.709984        0.611179            9053
10                    0.726561        0.597681            8108
11                    0.745230        0.602754            7552
12                    0.773942        0.581944            8518



TEST set precision, all predictions:
                              positives  precision
prediction                                        
pred0_manual_cci                    921   0.564604
pred1_manual_prev_g1              19158   0.578662
pred2_manual_prev_g1_and_snp      15244   0.572553
pred3_manual_dgs10_5              15910   0.587744
pred4_manual_dgs10_fedfunds       15921   0.502544
pred5_clf_10                       8108   0.597681
pred6_clf_best                     6677   0.629175

ANSWER Q4: best_max_depth = 4 (test precision = 0.6292)


## Question 5: What data is missing? (Exploratory)

Free-text: new indicators worth adding, and why.

In [6]:
"""
Question 5: [Exploratory] What data is missing?

Free-text answer: which new indicators would you add to the dataset, and why?
"""

ANSWER = """
The trees (Q4's depth-4 tree splits on cpi_core_yoy, FEDFUNDS, DGS10,
growth_btc_usd_365d and growth_wti_oil_365d first) lean almost entirely on
macro LEVELS. Those mostly identify the time period rather than the stock:
pred3/pred4 are 100%/0% positive on the whole validation window, and a
10-level tree beats depth 4 on train but loses on test. So the gaps are:

- Macro changes instead of levels -- 1/3/6-month deltas of DGS10, FEDFUNDS
  and CPI, plus the yield-curve slope (FRED T10Y2Y / T10Y3M). These carry
  the "rates are falling / curve is steepening" signal without memorising
  which years had 4% yields.
- Regional macro -- a third of the tickers are Indian or EU, but every macro
  feature is US. Add RBI repo rate + India CPI (RBI DBIE / FRED
  INDCPIALLMINMEI), ECB deposit rate + euro area HICP + Bund yield (ECB Data
  Portal / FRED IRLTLT01DEM156N), and the FX rates INR=X / EURUSD=X (yfinance)
  since a USD investor's return depends on them.
- Risk sentiment -- VIX (^VIX), India VIX (^INDIAVIX), VSTOXX, and the high-
  yield credit spread (FRED BAMLH0A0HYM2). Fear spikes tend to precede
  30-day rebounds, which is exactly the target.
- Relative strength -- each stock's growth_30d/90d MINUS its home index's
  (^NSEI, ^STOXX50E, ^GSPC), and its cross-sectional rank among the 33
  tickers on that date. These are about the stock, not the regime.
- Fundamentals/events -- days to next earnings date, last earnings surprise,
  forward P/E vs. its own history (yfinance .earnings_dates / .info, or
  Financial Modeling Prep for history), plus dividend/ex-dates.
- Seasonality, but compact -- Q1 shows Oct/Nov weeks carry the strongest
  calendar signal; a "days to month end" / "turn of month" flag and a
  "Q4 (Oct-Dec)" flag capture it in 2 columns instead of 60 sparse dummies.
"""


if __name__ == "__main__":
    print(ANSWER)


The trees (Q4's depth-4 tree splits on cpi_core_yoy, FEDFUNDS, DGS10,
growth_btc_usd_365d and growth_wti_oil_365d first) lean almost entirely on
macro LEVELS. Those mostly identify the time period rather than the stock:
pred3/pred4 are 100%/0% positive on the whole validation window, and a
10-level tree beats depth 4 on train but loses on test. So the gaps are:

- Macro changes instead of levels -- 1/3/6-month deltas of DGS10, FEDFUNDS
  and CPI, plus the yield-curve slope (FRED T10Y2Y / T10Y3M). These carry
  the "rates are falling / curve is steepening" signal without memorising
  which years had 4% yields.
- Regional macro -- a third of the tickers are Indian or EU, but every macro
  feature is US. Add RBI repo rate + India CPI (RBI DBIE / FRED
  INDCPIALLMINMEI), ECB deposit rate + euro area HICP + Bund yield (ECB Data
  Portal / FRED IRLTLT01DEM156N), and the FX rates INR=X / EURUSD=X (yfinance)
  since a USD investor's return depends on them.
- Risk sentiment -- VIX (^VIX), Indi

## Summary of Answers

| Question | Answer |
|---|---|
| Q1 — Max \|corr\| of a `month_wom` dummy | **0.025** (`October_w4`) |
| Q2 — Precision of best new hand rule | **0.588** (`pred3_manual_dgs10_5`) |
| Q3 — TEST rows where only `pred5_clf_10` is correct | **1219** |
| Q4 — Optimal tree depth (1–12) | **4** (test precision 0.629) |
| Q5 — Missing data | Macro deltas instead of levels, regional (India/EU) macro + FX, VIX/credit spreads, relative strength vs. home index, earnings events — see `q5.py` |